# Multilabel Hate Speech Classification (Classic ML)

Notebook ini berisi tahapan lengkap untuk membangun model klasifikasi multilabel ujaran kebencian menggunakan Machine Learning klasik:

- Logistic Regression (OVR)
- Linear SVM (OVR, calibrated)
- Classifier Chain Logistic Regression
- Classifier Chain SVM (Calibrated)
- Threshold tuning per-label
- Evaluasi menggunakan Exact Match, Hamming Loss, Micro-F1, Macro-F1
- Penyimpanan model (artifact) + threshold

Notebook digunakan untuk mendukung BAB IV (Hasil & Pembahasan) dan Lampiran Skripsi.

In [2]:
import time
import numpy as np
import pandas as pd
import pickle
import re
import os
from sklearn.metrics import classification_report, hamming_loss
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.multiclass import OneVsRestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from sklearn.multioutput import ClassifierChain

pd.set_option('display.max_colwidth', 120)

## Load Dataset
Menggunakan dataset hasil split yang telah dibuat pada tahap sebelumnya:
- train_df.csv
- test_df.csv

In [3]:
label_cols = [
    "HS_Individual", "HS_Group", "HS_Religion",
    "HS_Race", "HS_Physical", "HS_Gender"
]

train_df = pd.read_csv("/kaggle/input/datasethatespeechmultilabel/train_df.csv")
test_df  = pd.read_csv("/kaggle/input/datasethatespeechmultilabel/test_df.csv")

X_train = train_df["Tweet"].fillna("").astype(str)
y_train = train_df[label_cols]

X_test  = test_df["Tweet"].fillna("").astype(str)
y_test  = test_df[label_cols]

print("Train size:", len(train_df))
print("Test size :", len(test_df))

print("\nProporsi label train:")
print(y_train.mean())

print("\nProporsi label test:")
print(y_train.mean())

Train size: 10535
Test size : 2634

Proporsi label train:
HS_Individual    0.271476
HS_Group         0.150831
HS_Religion      0.060180
HS_Race          0.043000
HS_Physical      0.024490
HS_Gender        0.023256
dtype: float64

Proporsi label test:
HS_Individual    0.271476
HS_Group         0.150831
HS_Religion      0.060180
HS_Race          0.043000
HS_Physical      0.024490
HS_Gender        0.023256
dtype: float64


## TF-IDF Vectorization
- n-gram (1,2)
- min_df=3
- max_df=0.95
- max_features=30.000
- sublinear_tf=True

In [ ]:
vectorizer = TfidfVectorizer(
    ngram_range=(1,2),
    min_df=3,
    max_df=0.95,
    max_features=30000,
    sublinear_tf=True
)

vectorizer.fit(X_train)

X_train_vec = vectorizer.transform(X_train)
X_test_vec  = vectorizer.transform(X_test)

num_labels = y_train.shape[1]

## Load Saved Model (Optional)

Untuk fleksibilitas, set `USE_SAVED_MODEL = True` jika ingin menggunakan model yang sudah dilatih sebelumnya.

- **True**: Load model dan threshold dari file pickle yang sudah ada
- **False**: Training model dari awal (lanjut ke cell training di bawah)

File yang akan di-load: `/kaggle/input/model-baseline/scikitlearn/default/1/model_artifact_2.pkl`

In [5]:
# ========== KONFIGURASI: PILIH TRAINING ATAU LOAD MODEL ==========
USE_SAVED_MODEL = True  # Set True untuk load model, False untuk training ulang

if USE_SAVED_MODEL:
    print("="*80)
    print("LOADING SAVED MODEL...")
    print("="*80)
    
    # Path ke file pickle model yang sudah dilatih
    # Sesuaikan nama file dengan yang ada di folder saved_models/baseline/
    model_path = "/kaggle/input/model-baseline/scikitlearn/default/1/model_artifact.pkl"
    
    with open(model_path, "rb") as f:
        artifact = pickle.load(f)
    
    # Load semua komponen dari artifact
    vectorizer = artifact["vectorizer"]
    ovr_lr = artifact["ovr_lr"]
    ovr_svm = artifact["ovr_svm"]
    cc_lr = artifact["cc_lr"]
    cc_svm = artifact["cc_svm"]
    
    # Load thresholds dari nested dictionary 'thresh'
    best_thresh_ovr_lr = artifact["thresh"]["ovr_lr"]
    best_thresh_ovr_svm = artifact["thresh"]["ovr_svm"]
    best_thresh_cc_lr = artifact["thresh"]["cc_lr"]
    best_thresh_cc_svm = artifact["thresh"]["cc_svm"]
    
    label_cols = artifact["label_columns"]
    
    # Transform data test menggunakan vectorizer yang sudah di-load
    X_train_vec = vectorizer.transform(X_train)
    X_test_vec = vectorizer.transform(X_test)
    
    print(f"✓ Model loaded from: {model_path}")
    print(f"✓ Vectorizer loaded (vocab size: {len(vectorizer.get_feature_names_out())})")
    print(f"✓ Models loaded: OVR-LR, OVR-SVM, CC-LR, CC-SVM")
    print(f"✓ Thresholds loaded for all models")
    print(f"  - OVR-LR thresholds: {[round(t, 2) for t in best_thresh_ovr_lr]}")
    print(f"  - OVR-SVM thresholds: {[round(t, 2) for t in best_thresh_ovr_svm]}")
    print(f"  - CC-LR thresholds: {[round(t, 2) for t in best_thresh_cc_lr]}")
    print(f"  - CC-SVM thresholds: {[round(t, 2) for t in best_thresh_cc_svm]}")
    print("\nSKIP TRAINING CELLS - Langsung ke evaluasi/visualization!")
    print("="*80)
    
else:
    print("="*80)
    print("TRAINING MODE: Will train models from scratch")
    print("="*80)
    print("Continue to training cells below...")

LOADING SAVED MODEL...
✓ Model loaded from: /kaggle/input/model-baseline/scikitlearn/default/1/model_artifact.pkl
✓ Vectorizer loaded (vocab size: 13387)
✓ Models loaded: OVR-LR, OVR-SVM, CC-LR, CC-SVM
✓ Thresholds loaded for all models
  - OVR-LR thresholds: [0.5, 0.6, 0.58, 0.72, 0.76, 0.62]
  - OVR-SVM thresholds: [0.34, 0.3, 0.3, 0.32, 0.3, 0.16]
  - CC-LR thresholds: [0.5, 0.58, 0.62, 0.76, 0.8, 0.66]
  - CC-SVM thresholds: [0.34, 0.56, 0.2, 0.42, 0.48, 0.18]

SKIP TRAINING CELLS - Langsung ke evaluasi/visualization!


## Metric Functions (Exact Match, Hamming, Micro-F1, Macro-F1)

Empat metrik evaluasi yang digunakan:

- **Exact Match Ratio** — ketepatan sempurna semua label dalam satu sampel (sangat ketat)
- **Hamming Loss** — rata-rata kesalahan prediksi per label
- **Micro F1** — F1 dihitung secara global (agregat TP/FP/FN dari semua label)
- **Macro F1** — rata-rata F1 per label (setiap label berbobot sama)


In [6]:
def exact_match_ratio(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return np.mean(np.all(y_true == y_pred, axis=1))

def hamming_loss_(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return np.mean(y_true != y_pred)

def micro_f1(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    TP = np.sum((y_true == 1) & (y_pred == 1))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))

    precision = TP / (TP + FP + 1e-10)
    recall    = TP / (TP + FN + 1e-10)

    return 2 * precision * recall / (precision + recall + 1e-10)

def macro_f1(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    scores = []
    for j in range(y_true.shape[1]):
        yt = y_true[:, j]
        yp = y_pred[:, j]

        TP = np.sum((yt == 1) & (yp == 1))
        FP = np.sum((yt == 0) & (yp == 1))
        FN = np.sum((yt == 1) & (yp == 0))

        precision = TP/(TP+FP+1e-10)
        recall    = TP/(TP+FN+1e-10)
        f1 = 2*precision*recall/(precision+recall+1e-10)
        scores.append(f1)

    return np.mean(scores)

# Model Training (Baseline)

Model:
1. Logistic Regression (OVR)
2. Linear SVM + Calibrated (OVR)
3. Classifier Chain Logistic Regression
4. Classifier Chain SVM (Calibrated)

In [7]:
print("\n=== TRAINING: OVR Logistic Regression ===")

ovr_lr = OneVsRestClassifier(
    LogisticRegression(max_iter=2500, class_weight="balanced", random_state=42)
)

t0 = time.time()
ovr_lr.fit(X_train_vec, y_train)
print("Train time:", round(time.time()-t0,2), "s")

pred_ovr_lr = ovr_lr.predict(X_test_vec)

print("Exact Match:", exact_match_ratio(y_test, pred_ovr_lr))
print("Hamming Loss:", hamming_loss_(y_test, pred_ovr_lr))
print("Micro F1:", micro_f1(y_test, pred_ovr_lr))
print("Macro F1:", macro_f1(y_test, pred_ovr_lr))


=== TRAINING: OVR Logistic Regression ===
Train time: 3.04 s
Exact Match: 0.6583143507972665
Hamming Loss: 0.0743482662617059
Micro F1: 0.6583309100990029
Macro F1: 0.6004865661546478


### 1.2. OVR Calibrated SVM

Menggunakan `LinearSVC` dengan `CalibratedClassifierCV` (sigmoid) dalam skema One-vs-Rest agar menghasilkan probabilitas untuk threshold tuning.


In [8]:
print("\n=== TRAINING: OVR Calibrated SVM ===")

ovr_svm = OneVsRestClassifier(
    CalibratedClassifierCV(
        LinearSVC(class_weight="balanced", max_iter=5000),
        cv=3,
        method="sigmoid"
    )
)

t0 = time.time()
ovr_svm.fit(X_train_vec, y_train)
print("Train time:", round(time.time()-t0,2), "s")

pred_ovr_svm = ovr_svm.predict(X_test_vec)

print("Exact Match:", exact_match_ratio(y_test, pred_ovr_svm))
print("Hamming Loss:", hamming_loss_(y_test, pred_ovr_svm))
print("Micro F1:", micro_f1(y_test, pred_ovr_svm))
print("Macro F1:", macro_f1(y_test, pred_ovr_svm))


=== TRAINING: OVR Calibrated SVM ===
Train time: 1.34 s
Exact Match: 0.7293090356871678
Hamming Loss: 0.05998481397114654
Micro F1: 0.6296874999515653
Macro F1: 0.5636171975443888


### 1.3. Classifier Chain Logistic Regression

**Classifier Chain** mengubah multilabel menjadi rangkaian prediksi berantai: output label sebelumnya menjadi input untuk label berikutnya, sehingga menangkap ketergantungan antar label.


In [9]:
print("\n=== TRAINING: CC Logistic Regression ===")

cc_lr = ClassifierChain(
    LogisticRegression(max_iter=2500, class_weight="balanced", random_state=42),
    order="random",
    random_state=42
)

cc_lr.fit(X_train_vec, y_train)
pred_cc_lr = cc_lr.predict(X_test_vec)

print("Exact Match:", exact_match_ratio(y_test, pred_cc_lr))
print("Hamming Loss:", hamming_loss_(y_test, pred_cc_lr))
print("Micro F1:", micro_f1(y_test, pred_cc_lr))
print("Macro F1:", macro_f1(y_test, pred_cc_lr))


=== TRAINING: CC Logistic Regression ===
Exact Match: 0.6886864085041762
Hamming Loss: 0.08364970893444698
Micro F1: 0.6209862384829746
Macro F1: 0.5500909823180408


### 1.4. Classifier Chain Calibrated SVM

Kombinasi Classifier Chain dengan SVM yang sudah dikalibrasi (sigmoid), sehingga setiap estimator dalam rantai menghasilkan probabilitas.


In [10]:
print("\n=== TRAINING: CC Calibrated SVM ===")

cc_svm = ClassifierChain(
    CalibratedClassifierCV(
        LinearSVC(class_weight="balanced", max_iter=5000),
        cv=3,
        method="sigmoid"
    ),
    order="random",
    random_state=42
)

cc_svm.fit(X_train_vec, y_train)
probs_cc_svm = cc_svm.predict_proba(X_test_vec)
pred_cc_svm_default = (probs_cc_svm >= 0.5).astype(int)

print("Exact Match:", exact_match_ratio(y_test, pred_cc_svm_default))
print("Hamming Loss:", hamming_loss_(y_test, pred_cc_svm_default))
print("Micro F1:", micro_f1(y_test, pred_cc_svm_default))
print("Macro F1:", macro_f1(y_test, pred_cc_svm_default))


=== TRAINING: CC Calibrated SVM ===
Exact Match: 0.7589217919514047
Hamming Loss: 0.058402935965578336
Micro F1: 0.657004830868573
Macro F1: 0.6174056147692345


# Threshold Tuning

Setiap label dicari threshold terbaik pada rentang 0.1 — 0.9
berdasarkan skor **micro F1** validation set.

In [11]:
X_tr2, X_val2, y_tr2, y_val2 = train_test_split(
    X_train_vec, y_train, test_size=0.2, random_state=42
)

### 2.1. Manual Threshold Tuning: CC-SVM

Pertama, lakukan threshold tuning secara manual untuk model CC-SVM sebagai ilustrasi langkah per langkah.


In [12]:
print("\n=== Threshold Tuning: CC-SVM ===")
cc_svm.fit(X_tr2, y_tr2)
probs_val = cc_svm.predict_proba(X_val2)

best_thresh_cc_svm = []

for j in range(num_labels):
    best_score, best_t = 0, 0.5
    for t in np.linspace(0.1, 0.9, 41):
        pred_j = (probs_val[:, j] >= t).astype(int)
        score = micro_f1(y_val2.iloc[:, j], pred_j)
        if score > best_score:
            best_score, best_t = score, t
    best_thresh_cc_svm.append(best_t)

best_thresh_cc_svm


=== Threshold Tuning: CC-SVM ===


[0.33999999999999997, 0.56, 0.2, 0.42000000000000004, 0.48, 0.18]

### 2.2. Threshold Tuning Otomatis untuk Semua Model

Menggunakan fungsi `tune_threshold()` yang mencari threshold optimal per-label berdasarkan F1-score pada validation set.


In [13]:
from sklearn.metrics import f1_score
def tune_threshold(model, name):
    print(f"\n=== Threshold Tuning: {name} ===")
    model.fit(X_tr2, y_tr2)
    probs_val = model.predict_proba(X_val2)

    best = []
    for j in range(num_labels):
        best_score, best_t = 0, 0.5
        for t in np.linspace(0.1, 0.9, 41):
            pred_j = (probs_val[:, j] >= t).astype(int)
            
            # GUNAKAN f1_score standar karena ini evaluasi per satu label
            # Bukan fungsi macro_f1 buatan sendiri yang mengharapkan matriks banyak kolom
            score = f1_score(y_val2.iloc[:, j], pred_j, zero_division=0)
            
            if score > best_score:
                best_score, best_t = score, t
        best.append(best_t)
    return best

best_thresh_ovr_lr  = tune_threshold(ovr_lr, "OVR LR")
print(best_thresh_ovr_lr)

best_thresh_ovr_svm = tune_threshold(ovr_svm, "OVR SVM")
print(best_thresh_ovr_svm)

best_thresh_cc_lr   = tune_threshold(cc_lr, "CC LR")
print(best_thresh_cc_lr)

best_thresh_cc_svm   = tune_threshold(cc_svm, "CC SVM")
print(best_thresh_cc_svm)


=== Threshold Tuning: OVR LR ===
[0.5, 0.6, 0.58, 0.72, 0.76, 0.62]

=== Threshold Tuning: OVR SVM ===
[0.33999999999999997, 0.30000000000000004, 0.30000000000000004, 0.32, 0.30000000000000004, 0.16]

=== Threshold Tuning: CC LR ===
[0.5, 0.58, 0.62, 0.76, 0.8, 0.66]

=== Threshold Tuning: CC SVM ===
[0.33999999999999997, 0.56, 0.2, 0.42000000000000004, 0.48, 0.18]


## 3. Perbandingan Hasil Evaluasi

Membandingkan performa seluruh model (default vs tuned threshold) dalam satu tabel: Exact Match, Hamming Loss, Micro-F1, dan Macro-F1.


In [14]:
results = pd.DataFrame({
    "Model": [
        "OVR LR", "OVR LR (tuned)",
        "OVR SVM", "OVR SVM (tuned)",
        "CC LR", "CC LR (tuned)",
        "CC SVM", "CC SVM (tuned)"
    ],
    "Exact Match": [
        exact_match_ratio(y_test, pred_ovr_lr),
        exact_match_ratio(y_test, (ovr_lr.predict_proba(X_test_vec) >= best_thresh_ovr_lr).astype(int)),
        exact_match_ratio(y_test, pred_ovr_svm),
        exact_match_ratio(y_test, (ovr_svm.predict_proba(X_test_vec) >= best_thresh_ovr_svm).astype(int)),
        exact_match_ratio(y_test, pred_cc_lr),
        exact_match_ratio(y_test, (cc_lr.predict_proba(X_test_vec) >= best_thresh_cc_lr).astype(int)),
        exact_match_ratio(y_test, pred_cc_svm_default),
        exact_match_ratio(y_test, (probs_cc_svm >= best_thresh_cc_svm).astype(int)),
    ],
    "Hamming Loss": [
        hamming_loss_(y_test, pred_ovr_lr),
        hamming_loss_(y_test, (ovr_lr.predict_proba(X_test_vec) >= best_thresh_ovr_lr).astype(int)),
        hamming_loss_(y_test, pred_ovr_svm),
        hamming_loss_(y_test, (ovr_svm.predict_proba(X_test_vec) >= best_thresh_ovr_svm).astype(int)),
        hamming_loss_(y_test, pred_cc_lr),
        hamming_loss_(y_test, (cc_lr.predict_proba(X_test_vec) >= best_thresh_cc_lr).astype(int)),
        hamming_loss_(y_test, pred_cc_svm_default),
        hamming_loss_(y_test, (probs_cc_svm >= best_thresh_cc_svm).astype(int)),
    ],
    "Micro F1": [
        micro_f1(y_test, pred_ovr_lr),
        micro_f1(y_test, (ovr_lr.predict_proba(X_test_vec) >= best_thresh_ovr_lr).astype(int)),
        micro_f1(y_test, pred_ovr_svm),
        micro_f1(y_test, (ovr_svm.predict_proba(X_test_vec) >= best_thresh_ovr_svm).astype(int)),
        micro_f1(y_test, pred_cc_lr),
        micro_f1(y_test, (cc_lr.predict_proba(X_test_vec) >= best_thresh_cc_lr).astype(int)),
        micro_f1(y_test, pred_cc_svm_default),
        micro_f1(y_test, (probs_cc_svm >= best_thresh_cc_svm).astype(int)),
    ],
    "Macro F1": [
        macro_f1(y_test, pred_ovr_lr),
        macro_f1(y_test, (ovr_lr.predict_proba(X_test_vec) >= best_thresh_ovr_lr).astype(int)),
        macro_f1(y_test, pred_ovr_svm),
        macro_f1(y_test, (ovr_svm.predict_proba(X_test_vec) >= best_thresh_ovr_svm).astype(int)),
        macro_f1(y_test, pred_cc_lr),
        macro_f1(y_test, (cc_lr.predict_proba(X_test_vec) >= best_thresh_cc_lr).astype(int)),
        macro_f1(y_test, pred_cc_svm_default),
        macro_f1(y_test, (probs_cc_svm >= best_thresh_cc_svm).astype(int)),
    ]
})

results

,Model,Exact Match,Hamming Loss,Micro F1,Macro F1
0,OVR LR,0.658314,0.074348,0.658331,0.600487
1,OVR LR (tuned),0.697418,0.065490,0.664506,0.606500
2,OVR SVM,0.729309,0.059985,0.629687,0.563617
3,OVR SVM (tuned),0.692483,0.066819,0.661538,0.600815
4,CC LR,0.688686,0.083650,0.620986,0.550091
5,CC LR (tuned),0.723994,0.066755,0.657801,0.599066
6,CC SVM,0.758922,0.058403,0.657005,0.617406
7,CC SVM (tuned),0.733106,0.061060,0.675412,0.624143


## Classification Report Per Label

Menampilkan **Precision, Recall, F1-Score** untuk setiap label secara detail menggunakan `classification_report` dari scikit-learn.

Konfigurasi yang dibandingkan:
1. **CC-SVM** dengan threshold default (0.5)
2. **CC-SVM** dengan threshold tuned (optimized per-label)

In [18]:
from sklearn.metrics import precision_recall_fscore_support

# Prediksi untuk 2 konfigurasi CC-SVM
# 1. CC-SVM threshold default (0.5)
pred_cc_svm_default = (cc_svm.predict_proba(X_test_vec) >= 0.5).astype(int)

# 2. CC-SVM threshold tuned
pred_cc_svm_tuned = (cc_svm.predict_proba(X_test_vec) >= best_thresh_cc_svm).astype(int)

print("="*80)
print("1. CC-SVM (Threshold Default = 0.5)")
print("="*80)
print(classification_report(y_test, pred_cc_svm_default, target_names=label_cols, zero_division=0))

# Metrics per label dalam bentuk array
precision, recall, f1, support = precision_recall_fscore_support(
    y_test, pred_cc_svm_default, average=None, zero_division=0
)
print("\nPer-Label Metrics (CC-SVM Default):")
for i, label in enumerate(label_cols):
    print(f"{label:20s} | Precision: {precision[i]:.4f} | Recall: {recall[i]:.4f} | F1: {f1[i]:.4f} | Support: {support[i]}")

print("\n" + "="*80)
print("2. CC-SVM (Threshold Tuned)")
print("="*80)
print(f"Tuned thresholds: {[round(t, 2) for t in best_thresh_cc_svm]}")
print(classification_report(y_test, pred_cc_svm_tuned, target_names=label_cols, zero_division=0))

# Metrics per label dalam bentuk array
precision, recall, f1, support = precision_recall_fscore_support(
    y_test, pred_cc_svm_tuned, average=None, zero_division=0
)
print("\nPer-Label Metrics (CC-SVM Tuned):")
for i, label in enumerate(label_cols):
    print(f"{label:20s} | Precision: {precision[i]:.4f} | Recall: {recall[i]:.4f} | F1: {f1[i]:.4f} | Support: {support[i]}")

1. CC-SVM (Threshold Default = 0.5)
               precision    recall  f1-score   support

HS_Individual       0.77      0.62      0.69       715
     HS_Group       0.71      0.56      0.63       397
  HS_Religion       0.81      0.51      0.63       159
      HS_Race       0.78      0.63      0.70       113
  HS_Physical       0.71      0.42      0.52        65
    HS_Gender       0.59      0.36      0.45        61

    micro avg       0.75      0.57      0.65      1510
    macro avg       0.73      0.52      0.60      1510
 weighted avg       0.75      0.57      0.65      1510
  samples avg       0.25      0.24      0.25      1510


Per-Label Metrics (CC-SVM Default):
HS_Individual        | Precision: 0.7678 | Recall: 0.6196 | F1: 0.6858 | Support: 715
HS_Group             | Precision: 0.7102 | Recall: 0.5617 | F1: 0.6273 | Support: 397
HS_Religion          | Precision: 0.8100 | Recall: 0.5094 | F1: 0.6255 | Support: 159
HS_Race              | Precision: 0.7802 | Recall: 0.6283 | F

## Save Model + Threshold dalam satu file Pickle

In [16]:
save_dir = "saved_models"
os.makedirs(save_dir, exist_ok=True)

artifact = {
    "vectorizer": vectorizer,
    "ovr_lr": ovr_lr,
    "ovr_svm": ovr_svm,
    "cc_lr": cc_lr,
    "cc_svm": cc_svm,
    "thresh": {
        "ovr_lr": best_thresh_ovr_lr,
        "ovr_svm": best_thresh_ovr_svm,
        "cc_lr": best_thresh_cc_lr,
        "cc_svm": best_thresh_cc_svm
    },
    "label_columns": label_cols
}

with open(f"{save_dir}/model_artifact.pkl", "wb") as f:
    pickle.dump(artifact, f)

print("Model saved successfully.")

Model saved successfully.


## 4. Fungsi Prediksi (Inference)

Fungsi `predict_all()` untuk menguji model pada teks baru. Menggabungkan prediksi dari keempat model dengan threshold yang sudah di-tune.


In [17]:
def predict_all(text):
    vec = vectorizer.transform([text])

    p_lr  = (ovr_lr.predict_proba(vec)  >= best_thresh_ovr_lr ).astype(int)[0]
    p_svm = (ovr_svm.predict_proba(vec) >= best_thresh_ovr_svm).astype(int)[0]
    p_cc_lr  = (cc_lr.predict_proba(vec)  >= best_thresh_cc_lr ).astype(int)[0]
    p_cc_svm = (cc_svm.predict_proba(vec) >= best_thresh_cc_svm).astype(int)[0]

    return {
        "OVR_LR": dict(zip(label_cols, p_lr)),
        "OVR_SVM": dict(zip(label_cols, p_svm)),
        "CC_LR": dict(zip(label_cols, p_cc_lr)),
        "CC_SVM": dict(zip(label_cols, p_cc_svm)),
    }

predict_all("banci banget sih kelakuan lo")

{'OVR_LR': {'HS_Individual': 1,
  'HS_Group': 0,
  'HS_Religion': 0,
  'HS_Race': 0,
  'HS_Physical': 0,
  'HS_Gender': 1},
 'OVR_SVM': {'HS_Individual': 1,
  'HS_Group': 0,
  'HS_Religion': 0,
  'HS_Race': 0,
  'HS_Physical': 0,
  'HS_Gender': 1},
 'CC_LR': {'HS_Individual': 1,
  'HS_Group': 0,
  'HS_Religion': 0,
  'HS_Race': 0,
  'HS_Physical': 0,
  'HS_Gender': 1},
 'CC_SVM': {'HS_Individual': 1,
  'HS_Group': 0,
  'HS_Religion': 0,
  'HS_Race': 0,
  'HS_Physical': 0,
  'HS_Gender': 1}}

## 5. Kesimpulan

Ringkasan dari proses pemodelan baseline klasik:

1. **Empat model** berhasil dilatih: OVR-LR, OVR-SVM, CC-LR, dan CC-SVM.
2. **Threshold tuning** per-label meningkatkan F1-score pada beberapa label, terutama untuk label minoritas.
3. **CC-SVM** (Classifier Chain + SVM Calibrated) cenderung memberikan performa terbaik secara keseluruhan.
4. **Perbandingan** menunjukkan trade-off antara Exact Match (threshold default) dan F1-score (threshold tuned).
5. Model dan threshold telah disimpan dalam format pickle untuk digunakan pada tahap explainability dan deployment.
